## 1. Drive, paths

In [ ]:
import os
from huggingface_hub import login



HF_TOKEN = ""
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import glob
from pathlib import Path

# ---- EDIT IF NEEDED -------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive"
CKPT_HINT  = f"{DRIVE_ROOT}/best_encoder_mlm.pt"
TOK_HINT   = f"{DRIVE_ROOT}/legal_bpe_a1_50k_final.json"
OUT_DIR    = Path(f"{DRIVE_ROOT}/phase4_exp1_v2_encoder_full_ft")
# ---------------------------------------------------------------------------

def find_in_drive(hint, name):
    if Path(hint).exists():
        return Path(hint)
    hits = glob.glob(f"{DRIVE_ROOT}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found in Drive. Set the path above.")
    return Path(hits[0])

CKPT_DRIVE = find_in_drive(CKPT_HINT, "best_encoder_mlm.pt")
TOK_PATH   = find_in_drive(TOK_HINT,  "legal_bpe_a1_50k_final.json")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Phase-1 checkpoint:", CKPT_DRIVE)
print("Phase-1 tokenizer :", TOK_PATH)
print("Output directory  :", OUT_DIR)

Mounted at /content/drive
Phase-1 checkpoint: /content/drive/MyDrive/best_encoder_mlm.pt
Phase-1 tokenizer : /content/drive/MyDrive/legal_bpe_a1_50k_final.json
Output directory  : /content/drive/MyDrive/phase4_exp1_v2_encoder_full_ft


## 2. Imports, memory cap, hyper-parameters

## 3. Phase-1 tokenizer (unchanged)

In [4]:
N_SPECIAL, N_BYTES = 4, 256
BASE_VOCAB_SIZE = N_SPECIAL + N_BYTES
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

class A1Tokenizer:
    def __init__(self, itos, merges):
        self.itos = list(itos)
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos)
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}
        self._id_to_bytes = [b""] * N_SPECIAL + [bytes([b]) for b in range(256)]
        for l, r in self.merges:
            self._id_to_bytes.append(self._id_to_bytes[l] + self._id_to_bytes[r])
        assert len(self._id_to_bytes) == self.vocab_size
        self._cache = {}

    def _encode_word(self, word):                       # identical to Phase 1
        raw = word.encode("utf-8")
        seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks:
            return seq
        n = len(seq)
        prev = [-1] + list(range(0, n - 1))
        nxt = list(range(1, n)) + [-1]
        active = [True] * n
        heap = []
        for i in range(n - 1):
            rank = self._merge_ranks.get((seq[i], seq[i + 1]))
            if rank is not None:
                heapq.heappush(heap, (rank, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]:
                continue
            if nxt[left] != right or prev[right] != left:
                continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank:
                continue
            seq[left] = BASE_VOCAB_SIZE + rank
            active[right] = False
            right_next = nxt[right]
            nxt[left] = right_next
            if right_next != -1:
                prev[right_next] = left
            left_prev = prev[left]
            if left_prev != -1:
                new_rank = self._merge_ranks.get((seq[left_prev], seq[left]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left_prev, left))
            if right_next != -1:
                new_rank = self._merge_ranks.get((seq[left], seq[right_next]))
                if new_rank is not None:
                    heapq.heappush(heap, (new_rank, left, right_next))
        out, i = [], 0
        while i != -1:
            if active[i]:
                out.append(seq[i])
            i = nxt[i]
        return out

    def encode_normalized(self, text):
        # text: single-space separated, no leading/trailing space
        out, cache, first = [], self._cache, True
        for w in text.split(" "):
            if not first:
                out.append(N_SPECIAL + 32)
            first = False
            ids = cache.get(w)
            if ids is None:
                ids = self._encode_word(w)
                if len(cache) > 1_500_000:
                    cache.clear()
                cache[w] = ids
            out.extend(ids)
        return out

    def decode(self, ids):
        buf, chunks = bytearray(), []
        for i in ids:
            i = int(i)
            if i < N_SPECIAL:
                chunks.append(bytes(buf).decode("utf-8", errors="replace")); buf.clear()
                chunks.append(SPECIAL_TOKENS[i])
            else:
                buf.extend(self._id_to_bytes[i])
        chunks.append(bytes(buf).decode("utf-8", errors="replace"))
        return "".join(chunks)

with open(TOK_PATH, "r", encoding="utf-8") as f:
    _payload = json.load(f)
TOK = A1Tokenizer(_payload["itos"], _payload["merges"])
assert TOK.vocab_size == 50000 and TOK.itos[:4] == SPECIAL_TOKENS
assert N_SPECIAL + 32 == SPACE_ID

_t = "The Supreme Court held that the appeal was dismissed. भारत के संविधान के अनुसार।"
_ids = TOK.encode_normalized(_t)
assert TOK.decode(_ids) == _t
print("tokenizer ready | vocab", TOK.vocab_size, "| round-trip OK | space token id", SPACE_ID)

tokenizer ready | vocab 50000 | round-trip OK | space token id 36


## 4. Token cache v2 (lower-cased, compact, last 4,096 tokens)

Changes versus the first run's cache: text is **lower-cased** after whitespace normalisation; the single-space token is **removed at cache time**; only the **tail** is kept (the head was shown to be useless), but 4,096 compact tokens of it instead of 2,048. The original "has upper-case letters" flag is saved for error analysis. Test labels are never read.

In [5]:
SIG = f"v2_lower_compact_t{TAIL_TOK}_ch{TAIL_CH}"
CACHE_DIR = OUT_DIR / "token_cache"; CACHE_DIR.mkdir(exist_ok=True)
def cache_path(split): return CACHE_DIR / f"{split}_{SIG}.npz"
CACHE_READY = all(cache_path(s).exists() for s in SPLITS)
print("v2 token caches on Drive:", CACHE_READY)

_TEXTS = []

def _tok_job(i):
    raw = _TEXTS[i]
    no_upper = not any(c.isupper() for c in raw[:3000])
    s = " ".join(raw.lower().split()) or "."
    n = len(s)
    if n > TAIL_CH:
        s = s[-TAIL_CH:]
        cut = s.find(" ")
        s = s[cut + 1:] if cut >= 0 else s
    ids = np.asarray(TOK.encode_normalized(s), np.int64)
    ids = ids[ids != SPACE_ID]                        # compact representation
    full = (n <= TAIL_CH) and len(ids) <= TAIL_TOK
    return (ids[-TAIL_TOK:].astype(np.uint16), full, n, " ".join(raw.split())[:300], no_upper)

def tokenize_texts(texts):
    global _TEXTS
    _TEXTS = texts
    ctx = mp.get_context("fork")
    out, step = [], max(1, len(texts) // 10)
    with ctx.Pool(os.cpu_count()) as pool:
        for k, r in enumerate(pool.imap(_tok_job, range(len(texts)), chunksize=16), 1):
            out.append(r)
            if k % step == 0 or k == len(texts):
                print(f"   tokenised {k:,}/{len(texts):,}")
    _TEXTS = []
    return out

def load_json_records(path):
    with open(path, "r", encoding="utf-8") as f:
        first = f.read(1); f.seek(0)
        if first == "[":
            return json.load(f)
        return [json.loads(l) for l in f if l.strip()]

if not CACHE_READY:
    from huggingface_hub import hf_hub_download, login
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    if not HF_TOKEN:
        import getpass
        HF_TOKEN = getpass.getpass("Hugging Face token (read access): ")
    login(token=HF_TOKEN)
    REPO = "Exploration-Lab/CS-781-Capstone"
    FILES = {"train": "Phase_4_CJPE/Train/train.json",
             "validation": "Phase_4_CJPE/Train/dev.json",
             "test": "Phase_4_CJPE/Test/test.json"}
    DATA_DIR = Path("/content/cjpe_data"); DATA_DIR.mkdir(exist_ok=True)

    for split in SPLITS:
        if cache_path(split).exists():
            continue
        print(f"\n[{split}] downloading and loading")
        local = hf_hub_download(repo_id=REPO, filename=FILES[split], repo_type="dataset",
                                token=HF_TOKEN, local_dir=str(DATA_DIR))
        raw = load_json_records(local)
        seen, ids, texts, labels = set(), [], [], []
        for r in raw:
            cid = str(r["id"])
            if cid in seen:
                continue
            seen.add(cid); ids.append(cid); texts.append(r.get("text") or "")
            if split != "test":                      # test labels are never read
                labels.append(int(r["label"]))
        print(f"   rows {len(raw):,} -> unique cases {len(ids):,}")
        del raw; gc.collect()
        res = tokenize_texts(texts)
        del texts; gc.collect()

        off = np.zeros(len(res) + 1, np.int64)
        off[1:] = np.cumsum([len(r[0]) for r in res])
        payload = dict(ids=np.array(ids), tail_flat=np.concatenate([r[0] for r in res]), tail_off=off,
                       full=np.array([r[1] for r in res], bool),
                       nchars=np.array([r[2] for r in res], np.int64),
                       preview=np.array([r[3] for r in res]),
                       orig_no_upper=np.array([r[4] for r in res], bool))
        if split != "test":
            payload["labels"] = np.array(labels, np.int64)
        np.savez(cache_path(split), **payload)
        print("   saved", cache_path(split).name)
        del res, payload; gc.collect()
        os.remove(local)
print("token caches ready.")

v2 token caches on Drive: False
Hugging Face token (read access): ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.



[train] downloading and loading


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

   rows 37,387 -> unique cases 32,305
   tokenised 3,230/32,305
   tokenised 6,460/32,305
   tokenised 9,690/32,305
   tokenised 12,920/32,305
   tokenised 16,150/32,305
   tokenised 19,380/32,305
   tokenised 22,610/32,305
   tokenised 25,840/32,305
   tokenised 29,070/32,305
   tokenised 32,300/32,305
   tokenised 32,305/32,305
   saved train_v2_lower_compact_t4096_ch28000.npz

[validation] downloading and loading


... (2 outputs left out)

   rows 1,988 -> unique cases 994
   tokenised 99/994
   tokenised 198/994
   tokenised 297/994
   tokenised 396/994
   tokenised 495/994
   tokenised 594/994
   tokenised 693/994
   tokenised 792/994
   tokenised 891/994
   tokenised 990/994
   tokenised 994/994
   saved validation_v2_lower_compact_t4096_ch28000.npz

[test] downloading and loading


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

   rows 1,517 -> unique cases 1,517
   tokenised 151/1,517
   tokenised 302/1,517
   tokenised 453/1,517
   tokenised 604/1,517
   tokenised 755/1,517
   tokenised 906/1,517
   tokenised 1,057/1,517
   tokenised 1,208/1,517
   tokenised 1,359/1,517
   tokenised 1,510/1,517
   tokenised 1,517/1,517
   saved test_v2_lower_compact_t4096_ch28000.npz
token caches ready.


In [6]:
D = {}
for s in SPLITS:
    z = np.load(cache_path(s), allow_pickle=False)
    D[s] = {k: z[k] for k in z.files}
    D[s]["n"] = len(D[s]["ids"])
    assert len(set(D[s]["ids"].tolist())) == D[s]["n"]

def get_tail(s, i): d = D[s]; return d["tail_flat"][d["tail_off"][i]:d["tail_off"][i + 1]]
def label_of(s, i): return int(D[s]["labels"][i]) if "labels" in D[s] else -1
def n_windows(s, i):
    L = int(D[s]["tail_off"][i + 1] - D[s]["tail_off"][i])
    return min(K, max(1, -(-L // BODY)))

print(f"{'split':<12}{'cases':>8}{'pos rate':>10}{'whole doc kept':>16}{'avg windows':>13}{'orig. lower-case':>18}")
for s in SPLITS:
    d = D[s]
    pos = f"{d['labels'].mean():.3f}" if "labels" in d else "n/a"
    nw = np.array([n_windows(s, i) for i in range(d["n"])])
    print(f"{s:<12}{d['n']:>8,}{pos:>10}{100*d['full'].mean():>15.1f}%{nw.mean():>13.2f}{100*d['orig_no_upper'].mean():>17.1f}%")
assert max(int(D[s]["tail_flat"].max()) for s in SPLITS) < 50000
assert D["test"]["n"] == 1517 and "labels" not in D["test"]
assert all((get_tail(s, 0) != SPACE_ID).all() for s in SPLITS)
print("\nsample tail (decoded):", TOK.decode(get_tail("train", 0)[:40].tolist())[:160])

split          cases  pos rate  whole doc kept  avg windows  orig. lower-case
train         32,305     0.414           73.9%         4.96              0.0%
validation       994     0.500           64.3%         6.16            100.0%
test           1,517       n/a           62.6%         6.21             98.9%

sample tail (decoded): atholiccollege6wascompanycernedwithselectionandappointmentofaprincipalinanunaidedminorityeducationalinstitution.itwasstatedinpara19thattherightcompanyferredonmi


## 5. Model: Phase-1 encoder + document head

In [7]:
@dataclass
class ModelConfig:
    vocab_size: int = 50000
    d_model: int = 768
    n_layers: int = 16
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_seq_len: int = 512
    rope_base: float = 10000.0
    rms_norm_eps: float = 1e-6
    dropout: float = 0.0

def build_rope_cache(max_seq_len, d_head, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    ang = torch.cat([freqs, freqs], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    cos = cos.to(device=x.device, dtype=x.dtype)[None, None]
    sin = sin.to(device=x.device, dtype=x.dtype)[None, None]
    return x * cos + rotate_half(x) * sin

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class BidirectionalGQAAttention(nn.Module):
    # Phase-1 attention + an optional key-padding mask (Phase 1 never needed one: all windows were full)
    def __init__(self, c):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = c.d_model, c.n_heads, c.n_kv_heads
        self.head_dim = c.d_model // c.n_heads
        self.n_rep = c.n_heads // c.n_kv_heads
        self.max_seq_len = c.max_seq_len
        self.q_proj = nn.Linear(c.d_model, c.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(c.d_model, c.d_model, bias=False)
        cos, sin = build_rope_cache(c.max_seq_len, self.head_dim, c.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _split(self, x, h):
        B, L, _ = x.shape
        return x.reshape(B, L, h, self.head_dim).transpose(1, 2)

    def forward(self, x, key_mask=None):
        B, L, _ = x.shape
        q = self._split(self.q_proj(x), self.n_heads)
        k = self._split(self.k_proj(x), self.n_kv_heads)
        v = self._split(self.v_proj(x), self.n_kv_heads)
        q = apply_rope(q, self.rope_cos[:L], self.rope_sin[:L])
        k = apply_rope(k, self.rope_cos[:L], self.rope_sin[:L])
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        mask = None if key_mask is None else key_mask[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        return self.out_proj(a.transpose(1, 2).contiguous().view(B, L, self.d_model))


class EncoderBlock(nn.Module):
    def __init__(self, c, p_drop=0.0):
        super().__init__()
        self.attn_norm = RMSNorm(c.d_model, c.rms_norm_eps)
        self.attn = BidirectionalGQAAttention(c)
        self.ffn_norm = RMSNorm(c.d_model, c.rms_norm_eps)
        self.mlp = SwiGLU(c.d_model, c.d_ff)
        self.drop = nn.Dropout(p_drop)          # no parameters -> checkpoint-compatible
    def forward(self, x, key_mask=None):
        x = x + self.drop(self.attn(self.attn_norm(x), key_mask))
        return x + self.drop(self.mlp(self.ffn_norm(x)))

class LegalEncoder(nn.Module):
    # parameter names match the Phase-1 checkpoint (minus the tied mlm_head)
    def __init__(self, c, grad_ckpt=False, p_drop=0.0):
        super().__init__()
        self.grad_ckpt = grad_ckpt
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.layers = nn.ModuleList([EncoderBlock(c, p_drop) for _ in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)
    def forward(self, ids, key_mask=None):
        if key_mask is not None and bool(key_mask.all()):
            key_mask = None
        x = self.token_embedding(ids)
        for layer in self.layers:
            if self.grad_ckpt and self.training:
                x = checkpoint(layer, x, key_mask, use_reentrant=False)
            else:
                x = layer(x, key_mask)
        return self.final_norm(x)

class LegalDocClassifier(nn.Module):
    # window = mean-pooled encoder output; document = attention over windows (+ distance-from-end embedding),
    # concatenated with the last window's vector, then a linear classifier.
    def __init__(self, c, max_chunks=K, resid_drop=0.0, head_drop=0.1, grad_ckpt=False):
        super().__init__()
        d = c.d_model
        self.max_chunks = max_chunks
        self.encoder = LegalEncoder(c, grad_ckpt, resid_drop)
        self.chunk_pos = nn.Embedding(max_chunks, d)
        nn.init.normal_(self.chunk_pos.weight, std=0.02)
        self.chunk_attn = nn.Sequential(nn.Linear(d, 256), nn.Tanh(), nn.Linear(256, 1, bias=False))
        self.dropout = nn.Dropout(head_drop)
        self.classifier = nn.Linear(2 * d, 2)
        nn.init.normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)
        self.last_chunk_attn, self.last_chunk_mask = None, None

    def forward(self, batch):
        am = batch["attn_mask"]
        h = self.encoder(batch["input_ids"], am).float()
        m = am.unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1).clamp(min=1)                       # [N, d]
        B, Kc, d = batch["n_docs"], self.max_chunks, v.shape[-1]
        di, sl = batch["doc_index"], batch["slot"]
        chunks = v.new_zeros(B, Kc, d).index_put((di, sl), v)
        cm = torch.zeros(B, Kc, dtype=torch.bool, device=v.device)
        cm[di, sl] = True
        pe = self.chunk_pos(torch.arange(Kc - 1, -1, -1, device=v.device))   # slot s -> distance Kc-1-s from the end
        sc = self.chunk_attn(chunks + pe.unsqueeze(0)).squeeze(-1).float().masked_fill(~cm, float("-inf"))
        w = torch.softmax(sc, dim=-1)
        agg = (w.unsqueeze(-1) * chunks).sum(1)
        doc = torch.cat([agg, chunks[:, Kc - 1]], dim=-1)               # last chunk always present
        self.last_chunk_attn, self.last_chunk_mask = w.detach(), cm
        return self.classifier(self.dropout(doc))

LOCAL_CKPT = Path("/content/best_encoder_mlm.pt")
if not LOCAL_CKPT.exists():
    shutil.copy(CKPT_DRIVE, LOCAL_CKPT)
P1 = torch.load(LOCAL_CKPT, map_location="cpu", weights_only=True, mmap=True)
P1_CFG = ModelConfig(**P1["model_config"])
print("Phase-1 checkpoint:", P1.get("architecture"), "| step", P1.get("global_step"),
      "| best val MLM loss", P1.get("best_val_mlm_loss"))

def load_phase1_weights(encoder):
    missing, unexpected = encoder.load_state_dict(P1["model_state_dict"], strict=False)
    assert not missing, f"missing keys: {missing}"
    assert set(unexpected) <= {"mlm_head.weight"}, f"unexpected keys: {unexpected}"

def build_model(pretrained=True):
    m = LegalDocClassifier(P1_CFG, K, RESID_DROPOUT, HEAD_DROPOUT, GRAD_CKPT)
    if pretrained:
        load_phase1_weights(m.encoder)
    return m.to(device)

def model_from_state(state):
    m = build_model(pretrained=False)
    m.load_state_dict(state)
    return m.eval()

Phase-1 checkpoint: bidirectional_gqa_encoder | step 15000 | best val MLM loss 1.0438882003729564


## 6. Document → windows, micro-batch packing

Windows are the **last K = 8 chunks** of the compact, lower-cased document (`<s>` + 511 tokens each, oldest chunk possibly shorter). Short documents simply have fewer windows. During training each non-final chunk is dropped with probability 0.1 (augmentation that also saves compute). Documents are packed greedily into micro-batches of at most 16 windows, which keeps peak memory far below the 14 GB cap without gradient checkpointing.

In [8]:
def build_chunks(split, i, drop_p=0.0, rng=None):
    tail = get_tail(split, i).astype(np.int64)
    body, end = [], len(tail)
    while end > 0 and len(body) < K:
        start = max(0, end - BODY)
        body.append(tail[start:end]); end = start
    body = body[::-1] if body else [tail]
    n = len(body)
    keep = [True] * n
    if drop_p > 0 and n > 1:
        for j in range(n - 1):                         # never drop the final chunk
            if rng.random() < drop_p:
                keep[j] = False
    return [(K - n + j, np.concatenate([[BOS_ID], c]).astype(np.int64)) for j, c in enumerate(body) if keep[j]]

def collate(items, train=False, rng=None):
    seqs, didx, slot = [], [], []
    for b, (s, i) in enumerate(items):
        for sl, c in build_chunks(s, i, CHUNK_DROP if train else 0.0, rng):
            seqs.append(c); didx.append(b); slot.append(sl)
    L = max(len(c) for c in seqs)
    ids = np.zeros((len(seqs), L), np.int64); am = np.zeros((len(seqs), L), bool)
    for r, c in enumerate(seqs):
        ids[r, :len(c)] = c; am[r, :len(c)] = True
    return dict(input_ids=torch.from_numpy(ids), attn_mask=torch.from_numpy(am),
                doc_index=torch.tensor(didx, dtype=torch.long), slot=torch.tensor(slot, dtype=torch.long),
                n_docs=len(items))

def pack(items, max_windows):
    groups, cur, w = [], [], 0
    for it in items:
        n = n_windows(*it)
        if cur and w + n > max_windows:
            groups.append(cur); cur, w = [], 0
        cur.append(it); w += n
    if cur:
        groups.append(cur)
    return groups

def to_dev(b):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}

# sanity: Phase-1 weights + lower-cased compact tokens are coherent (MLM loss far below ln(50000) = 10.82)
@torch.inference_mode()
def mlm_sanity(n_docs=16):
    enc = LegalEncoder(P1_CFG).to(device).eval(); load_phase1_weights(enc)
    g = torch.Generator().manual_seed(0)
    b = collate([("train", i) for i in range(n_docs)])
    ids, am = b["input_ids"].to(device), b["attn_mask"].to(device)
    sel = (torch.rand(ids.shape, generator=g) < 0.15).to(device) & am & (ids > 3)
    with torch.autocast("cuda", dtype=AMP_DTYPE):
        logits = F.linear(enc(ids.masked_fill(sel, UNK_ID), am), enc.token_embedding.weight)
    loss = float(F.cross_entropy(logits[sel].float(), ids[sel]))
    del enc; torch.cuda.empty_cache()
    return loss
print(f"Phase-1 MLM loss on lower-cased compact CJPE windows: {mlm_sanity():.3f}")

Phase-1 MLM loss on lower-cased compact CJPE windows: 4.766


## 7. Training utilities (EMA weights, half-epoch evaluation, memory tracking)

In [9]:
def tune_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    f = np.array([f1_score(y, (p >= t).astype(int), average="macro") for t in grid])
    sm = np.convolve(np.pad(f, 2, mode="edge"), np.ones(5) / 5, mode="valid")
    k = int(np.argmax(sm)); return float(grid[k]), float(f[k])

def prior_threshold(p, rate):
    # threshold that labels the top `rate` fraction of documents as class 1
    return float(np.quantile(p, 1.0 - rate))

def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    return dict(macro_f1=float(f1_score(y, pred, average="macro")),
                precision_macro=float(precision_score(y, pred, average="macro", zero_division=0)),
                recall_macro=float(recall_score(y, pred, average="macro", zero_division=0)),
                f1_per_class=[float(x) for x in f1_score(y, pred, average=None)],
                recall_per_class=[float(x) for x in recall_score(y, pred, average=None, zero_division=0)],
                pred_pos_rate=float(pred.mean()))

@torch.inference_mode()
def predict_proba(model, items):
    was = model.training; model.eval()
    nw = np.array([n_windows(s, i) for s, i in items])
    order = np.argsort(-nw, kind="stable")
    ordered = [items[j] for j in order]
    out = np.zeros(len(items)); asum, acnt = np.zeros(K), np.zeros(K); pos = 0
    for g in pack(ordered, EVAL_WINDOWS):
        batch = to_dev(collate(g))
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(batch)
        out[order[pos:pos + len(g)]] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        pos += len(g)
        w, cm = model.last_chunk_attn.cpu().numpy(), model.last_chunk_mask.cpu().numpy()
        asum += (w * cm).sum(0); acnt += cm.sum(0)
    model.train(was)
    return out, (asum / np.maximum(acnt, 1)).tolist()

class EMA:
    def __init__(self, model, decay):
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0
    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_mul_(list(self.model.parameters()), d)
        torch._foreach_add_(list(self.model.parameters()), [p.detach() for p in model.parameters()], alpha=1 - d)
    def state(self):
        return {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}

def build_optimizer(model):
    L = P1_CFG.n_layers
    groups = {}
    for name, p in model.named_parameters():
        if name.startswith("encoder."):
            if name.startswith("encoder.token_embedding"): depth = L
            elif name.startswith("encoder.layers."):        depth = L - 1 - int(name.split(".")[2])
            else:                                            depth = 0
            lr = ENC_LR * (LLRD ** depth)
        else:
            lr = HEAD_LR
        wd = 0.0 if (p.ndim == 1 or "token_embedding" in name) else WD
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([dict(params=ps, lr=lr, weight_decay=wd) for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)

def train_model(train_items, eval_items=None, stop_step=None, label=""):
    # LR schedule always spans E_MAX epochs; stop_step lets Stage B replicate Stage A up to its best step.
    set_seed(SEED); torch.cuda.reset_peak_memory_stats()
    model = build_model(); model.train()
    ema = EMA(model, EMA_DECAY)
    y_tr = np.array([label_of(s, i) for s, i in train_items])
    n0, n1 = int((y_tr == 0).sum()), int((y_tr == 1).sum())
    cw = torch.tensor([len(y_tr) / (2 * n0), len(y_tr) / (2 * n1)], dtype=torch.float32, device=device)
    steps_per_epoch = DPE // EFF_DOCS
    total = E_MAX * steps_per_epoch
    run_steps = min(stop_step or total, total)
    warm = max(1, int(WARMUP * total))
    opt = build_optimizer(model)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    rng = np.random.default_rng(SEED); crng = np.random.default_rng(SEED + 1)
    order, ptr = rng.permutation(len(train_items)), 0
    y_ev = np.array([label_of(s, i) for s, i in eval_items]) if eval_items else None
    rate = TARGET_POS_RATE if TARGET_POS_RATE is not None else (float(y_ev.mean()) if y_ev is not None else 0.5)

    print(f"[{label}] documents {len(train_items):,} | steps {run_steps} of {total} | class weights {[round(x,3) for x in cw.tolist()]}")
    history, best, best_probs, best_attn, best_state = [], -1.0, None, None, None
    loss_acc, n_acc = 0.0, 0
    for step in range(1, run_steps + 1):
        take = []
        for _ in range(EFF_DOCS):
            if ptr >= len(order):
                order, ptr = rng.permutation(len(train_items)), 0
            take.append(train_items[int(order[ptr])]); ptr += 1
        opt.zero_grad(set_to_none=True)
        for g in pack(take, MICRO_WINDOWS):
            batch = to_dev(collate(g, train=True, rng=crng))
            yb = torch.tensor([label_of(s, i) for s, i in g], device=device)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(batch)
            loss = F.cross_entropy(logits.float(), yb, weight=cw, reduction="sum",
                                   label_smoothing=LABEL_SMOOTH) / EFF_DOCS
            scaler.scale(loss).backward()
            loss_acc += float(loss.detach())
        n_acc += 1
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scaler.step(opt); scaler.update(); sched.step()
        ema.update(model)

        if eval_items is not None and (step % EVAL_EVERY == 0 or step == run_steps):
            p, attn = predict_proba(ema.model, eval_items)
            thr, f_thr = tune_threshold(y_ev, p)
            rec = dict(step=step, epoch=step / steps_per_epoch, train_loss=loss_acc / n_acc,
                       auc=float(roc_auc_score(y_ev, p)),
                       f1_at_0p5=float(f1_score(y_ev, (p >= 0.5).astype(int), average="macro")),
                       tuned_threshold=thr, f1_at_tuned=f_thr,
                       f1_at_prior=float(f1_score(y_ev, (p >= prior_threshold(p, rate)).astype(int), average="macro")))
            history.append(rec)
            print(f"[{label}] step {step:>5} (epoch {rec['epoch']:.1f}) | train loss {rec['train_loss']:.4f} | dev AUC {rec['auc']:.4f} | "
                  f"F1@0.5 {rec['f1_at_0p5']:.4f} | F1@tuned({thr:.2f}) {f_thr:.4f} | F1@top-{rate:.0%} {rec['f1_at_prior']:.4f}")
            loss_acc, n_acc = 0.0, 0
            if rec["auc"] > best:
                best, best_probs, best_attn = rec["auc"], p.copy(), attn
                best_state = ema.state(); best_step = step
    out = dict(history=history, peak_mem_gb=torch.cuda.max_memory_allocated() / 1024**3,
               best_step=(best_step if eval_items is not None else run_steps), best_auc=best,
               best_probs=best_probs, best_chunk_attn=best_attn, best_state=best_state,
               final_state=ema.state())
    del model, ema, opt, sched; gc.collect(); torch.cuda.empty_cache()
    return out

train_items_all = [("train", i) for i in range(D["train"]["n"])]
dev_items = [("validation", i) for i in range(D["validation"]["n"])]
test_items = [("test", i) for i in range(D["test"]["n"])]
y_dev = np.array([label_of(s, i) for s, i in dev_items])
print("ready | train", len(train_items_all), "| dev", len(dev_items), "| test", len(test_items))

ready | train 32305 | dev 994 | test 1517


## 8. Stage A — train on the (lower-cased) training set, validate on dev every half epoch

Reference from run 1 (cased train, 4 chunks, no EMA): dev AUC **0.8346**, macro-F1 **0.7517**. The best step is chosen by dev AUC of the EMA weights.

In [10]:
A_JSON, A_PROBS, A_STATE = OUT_DIR / "stageA_summary.json", OUT_DIR / "stageA_dev_probs.npy", OUT_DIR / "stageA_best_state.pt"
if A_JSON.exists() and A_PROBS.exists() and A_STATE.exists():
    A = json.load(open(A_JSON)); A["best_probs"] = np.load(A_PROBS)
    A["best_state"] = torch.load(A_STATE, map_location="cpu")
    print("Stage A loaded from Drive (delete stageA_* files to re-run).")
else:
    A = train_model(train_items_all, eval_items=dev_items, label="stage A")
    np.save(A_PROBS, A["best_probs"]); torch.save(A["best_state"], A_STATE)
    json.dump({k: v for k, v in A.items() if k not in ("best_probs", "best_state", "final_state")}, open(A_JSON, "w"), indent=1)
BEST_STEP = int(A["best_step"]); dev_probs = A["best_probs"]
print(f"\nbest step {BEST_STEP} (epoch {BEST_STEP/(DPE//EFF_DOCS):.1f}) | dev AUC {A['best_auc']:.4f} | peak GPU memory {A['peak_mem_gb']:.1f} GB")

[stage A] documents 32,305 | steps 4000 of 4000 | class weights [0.854, 1.207]
[stage A] step   500 (epoch 0.5) | train loss 0.7737 | dev AUC 0.6588 | F1@0.5 0.6029 | F1@tuned(0.47) 0.6176 | F1@top-50% 0.6016
[stage A] step  1000 (epoch 1.0) | train loss 0.6594 | dev AUC 0.7257 | F1@0.5 0.6526 | F1@tuned(0.47) 0.6578 | F1@top-50% 0.6660
[stage A] step  1500 (epoch 1.5) | train loss 0.6140 | dev AUC 0.7854 | F1@0.5 0.6882 | F1@tuned(0.43) 0.7193 | F1@top-50% 0.7163
[stage A] step  2000 (epoch 2.0) | train loss 0.5828 | dev AUC 0.8259 | F1@0.5 0.7146 | F1@tuned(0.39) 0.7545 | F1@top-50% 0.7545
[stage A] step  2500 (epoch 2.5) | train loss 0.5296 | dev AUC 0.8356 | F1@0.5 0.7359 | F1@tuned(0.36) 0.7592 | F1@top-50% 0.7565
[stage A] step  3000 (epoch 3.0) | train loss 0.5022 | dev AUC 0.8452 | F1@0.5 0.7471 | F1@tuned(0.42) 0.7705 | F1@top-50% 0.7746
[stage A] step  3500 (epoch 3.5) | train loss 0.4853 | dev AUC 0.8443 | F1@0.5 0.7415 | F1@tuned(0.37) 0.7736 | F1@top-50% 0.7726
[stage A] s

## 9. Dev analysis (for the report)

In [11]:
y, p = y_dev, dev_probs
rate = TARGET_POS_RATE if TARGET_POS_RATE is not None else float(y.mean())
thr_dev, _ = tune_threshold(y, p)
thr_prior = prior_threshold(p, rate)
rows = [("threshold 0.50", 0.5), (f"dev-tuned ({thr_dev:.2f})", thr_dev), (f"top-{rate:.0%} ({thr_prior:.2f})", thr_prior)]
print(f"{'':<24}{'macro-F1':>10}{'macro-P':>10}{'macro-R':>10}{'F1 cls0':>10}{'F1 cls1':>10}{'R cls0':>9}{'R cls1':>9}{'pred+':>8}")
for name, t in rows:
    m = metrics_at(y, p, t)
    print(f"{name:<24}{m['macro_f1']:>10.4f}{m['precision_macro']:>10.4f}{m['recall_macro']:>10.4f}"
          f"{m['f1_per_class'][0]:>10.4f}{m['f1_per_class'][1]:>10.4f}{m['recall_per_class'][0]:>9.4f}{m['recall_per_class'][1]:>9.4f}{m['pred_pos_rate']:>8.3f}")
print(f"ROC-AUC {roc_auc_score(y, p):.4f}   (run 1: 0.8346 / macro-F1 0.7517)")

pred = (p >= thr_prior).astype(int); d = D["validation"]
print("\nconfusion matrix (rows = truth, cols = prediction):\n", confusion_matrix(y, pred))
fp, fn, ok = (pred == 1) & (y == 0), (pred == 0) & (y == 1), pred == y
print(f"\n{'group':<16}{'count':>7}{'median chars':>14}{'mean P(1)':>11}")
for name, mk in [("correct", ok), ("false positive", fp), ("false negative", fn)]:
    print(f"{name:<16}{int(mk.sum()):>7}{np.median(d['nchars'][mk]):>14,.0f}{p[mk].mean():>11.3f}")
q = np.quantile(d["nchars"], [0, .25, .5, .75, 1.0])
print("\nerror rate by length quartile:")
for lo, hi in zip(q[:-1], q[1:]):
    mk = (d["nchars"] >= lo) & (d["nchars"] <= hi)
    print(f"  {lo:>9,.0f} - {hi:>9,.0f} chars: {100*(1-ok[mk].mean()):5.1f}%  (n={int(mk.sum())})")
nw = np.array([n_windows("validation", i) for i in range(d["n"])])
print("\nerror rate by number of windows used:")
for lo, hi in [(1, 2), (3, 5), (6, 7), (8, 8)]:
    mk = (nw >= lo) & (nw <= hi)
    if mk.any(): print(f"  {lo}-{hi} windows: {100*(1-ok[mk].mean()):5.1f}%  (n={int(mk.sum())})")
print("\nmean document-attention per chunk (oldest -> last):", [round(x, 3) for x in A["best_chunk_attn"]])
print("\nlearning curve (EMA weights):")
for r in A["history"]:
    print(f"  step {r['step']:>5}  AUC {r['auc']:.4f}  F1@top-{rate:.0%} {r['f1_at_prior']:.4f}")

                          macro-F1   macro-P   macro-R   F1 cls0   F1 cls1   R cls0   R cls1   pred+
threshold 0.50              0.7471    0.7649    0.7505    0.7766    0.7175   0.8672   0.6338   0.383
dev-tuned (0.42)            0.7705    0.7713    0.7706    0.7760    0.7649   0.7948   0.7465   0.476
top-50% (0.40)              0.7746    0.7746    0.7746    0.7746    0.7746   0.7746   0.7746   0.500
ROC-AUC 0.8452   (run 1: 0.8346 / macro-F1 0.7517)

confusion matrix (rows = truth, cols = prediction):
 [[385 112]
 [112 385]]

group             count  median chars  mean P(1)
correct             770        16,201      0.443
false positive      112        19,604      0.567
false negative      112        18,158      0.254

error rate by length quartile:
        894 -    10,649 chars:  17.3%  (n=249)
     10,649 -    16,656 chars:  22.6%  (n=248)
     16,656 -    26,466 chars:  23.8%  (n=248)
     26,466 -   259,571 chars:  26.5%  (n=249)

error rate by number of windows used:
  1-2 window

## 10. Stage B — retrain on `train + dev×3` and stop at Stage A's best step

Same seed, same LR schedule, same EMA — it simply stops at the step that was best on dev, so the validated trajectory is reproduced with the dev documents added (repeated 3×, because dev has the same style as test).

In [12]:
B_STATE = OUT_DIR / "stageB_state.pt"
if B_STATE.exists():
    B = dict(final_state=torch.load(B_STATE, map_location="cpu"), peak_mem_gb=float("nan"))
    print("Stage B loaded from Drive (delete stageB_state.pt to re-run).")
else:
    final_items = train_items_all + dev_items * DEV_UPSAMPLE
    B = train_model(final_items, eval_items=None, stop_step=BEST_STEP, label="stage B")
    torch.save(B["final_state"], B_STATE)
print("Stage B ready | peak GPU memory:", B["peak_mem_gb"])

[stage B] documents 35,287 | steps 3000 of 4000 | class weights [0.864, 1.186]
Stage B ready | peak GPU memory: 8.356887340545654


## 11. Test predictions, submission file, model files

In [13]:
from scipy.stats import rankdata

mA = model_from_state(A["best_state"]); pA, _ = predict_proba(mA, test_items); del mA
mB = model_from_state(B["final_state"]); pB, _ = predict_proba(mB, test_items)
np.save(OUT_DIR / "test_probs_stageA.npy", pA); np.save(OUT_DIR / "test_probs_stageB.npy", pB)

def rank01(x): return (rankdata(x) - 1) / (len(x) - 1)
score = (rank01(pA) + rank01(pB)) / 2 if USE_ENSEMBLE else pB
rate = TARGET_POS_RATE if TARGET_POS_RATE is not None else float(y_dev.mean())
thr = prior_threshold(score, rate)
test_pred = (score >= thr).astype(int)

print(f"agreement between Stage A and Stage B predictions: {((pA >= 0.5) == (pB >= 0.5)).mean():.3f}")
print(f"positive rate at 0.5 -> A {np.mean(pA >= 0.5):.3f} | B {np.mean(pB >= 0.5):.3f}  (dev truth {y_dev.mean():.2f}; run 1 gave 0.625)")
print(f"final rule: top {rate:.0%} of the {'rank-averaged ensemble' if USE_ENSEMBLE else 'Stage B'} scores -> class 1 | predicted positives {test_pred.mean():.3f}")

test_ids = [str(x) for x in D["test"]["ids"]]
SUBMISSION_PATH = OUT_DIR / "submission_exp1_v2_encoder_full_ft.json"
json.dump([{"id": cid, "pred": int(pr)} for cid, pr in zip(test_ids, test_pred)], open(SUBMISSION_PATH, "w"), indent=2)

common = dict(encoder_config=asdict(P1_CFG), K=K, resid_dropout=RESID_DROPOUT, head_dropout=HEAD_DROPOUT,
              preprocessing="whitespace-normalised, lower-cased, space tokens removed, last 4096 tokens, 8 chunks of 511",
              window=WIN, special_ids=dict(pad=PAD_ID, unk=UNK_ID, bos=BOS_ID, eos=EOS_ID))
MODEL_PATH = OUT_DIR / "exp1_v2_encoder_full_ft_model.pt"
torch.save(dict(model_state_dict=B["final_state"], model_state_dict_stageA=A["best_state"],
                trained_on="train + dev x3 (stage B); stage A = train only",
                final_rule=f"rank-average(A,B), top {rate:.0%} -> class 1" if USE_ENSEMBLE else "stage B, top rate",
                best_step=BEST_STEP, dev_auc_stageA=A["best_auc"], **common), MODEL_PATH)
print("saved:", SUBMISSION_PATH); print("saved:", MODEL_PATH)

agreement between Stage A and Stage B predictions: 0.831
positive rate at 0.5 -> A 0.399 | B 0.500  (dev truth 0.50; run 1 gave 0.625)
final rule: top 50% of the rank-averaged ensemble scores -> class 1 | predicted positives 0.500
saved: /content/drive/MyDrive/phase4_exp1_v2_encoder_full_ft/submission_exp1_v2_encoder_full_ft.json
saved: /content/drive/MyDrive/phase4_exp1_v2_encoder_full_ft/exp1_v2_encoder_full_ft_model.pt


## 12. Validate the submission and report cost / memory

In [14]:
sub = json.load(open(SUBMISSION_PATH))
assert isinstance(sub, list) and len(sub) == len(test_ids) == 1517
assert [r["id"] for r in sub] == test_ids and len({r["id"] for r in sub}) == len(sub)
assert all(set(r) == {"id", "pred"} and type(r["pred"]) is int and r["pred"] in (0, 1) for r in sub)
print("submission OK:", len(sub), "predictions | class counts:", {c: sum(r["pred"] == c for r in sub) for c in (0, 1)})

n_total = sum(p_.numel() for p_ in mB.parameters())
n_emb = mB.encoder.token_embedding.weight.numel()
avg_w = float(np.mean([n_windows("test", i) for i in range(D["test"]["n"])]))
peak = max(A["peak_mem_gb"], B["peak_mem_gb"]) if B["peak_mem_gb"] == B["peak_mem_gb"] else A["peak_mem_gb"]
cost = dict(total_params=n_total, embedding_params=n_emb, fp32_size_mb=n_total * 4 / 1024**2,
            fp16_size_mb=n_total * 2 / 1024**2, avg_windows_per_test_doc=avg_w,
            forward_gflops_per_avg_doc=2 * (n_total - n_emb) * WIN * avg_w / 1e9,
            peak_train_gpu_gb=peak, memory_cap_gb=MEM_CAP_GB)
json.dump(cost, open(OUT_DIR / "cost_exp1_v2.json", "w"), indent=1)
for k, v in cost.items():
    print(f"  {k:<28}{v:>16,.2f}" if isinstance(v, float) else f"  {k:<28}{v:>16,}")
assert peak <= MEM_CAP_GB, "peak memory above the cap"
print(f"\npeak training memory {peak:.1f} GB <= {MEM_CAP_GB} GB cap (fits a free 15 GB T4)")
print("GPU used for this run:", gpu_name)
for p_ in sorted(OUT_DIR.glob("*")):
    if p_.is_file():
        print(f"  {p_.name:<46}{p_.stat().st_size/1024**2:>9.1f} MB")

submission OK: 1517 predictions | class counts: {0: 758, 1: 759}
  total_params                     139,294,978
  embedding_params                  38,400,000
  fp32_size_mb                          531.37
  fp16_size_mb                          265.68
  avg_windows_per_test_doc                6.21
  forward_gflops_per_avg_doc            641.69
  peak_train_gpu_gb                       8.36
  memory_cap_gb                          14.00

peak training memory 8.4 GB <= 14.0 GB cap (fits a free 15 GB T4)
GPU used for this run: NVIDIA A100-SXM4-40GB
  cost_exp1_v2.json                                   0.0 MB
  exp1_v2_encoder_full_ft_model.pt                 1062.8 MB
  stageA_best_state.pt                              531.4 MB
  stageA_dev_probs.npy                                0.0 MB
  stageA_summary.json                                 0.0 MB
  stageB_state.pt                                   531.4 MB
  submission_exp1_v2_encoder_full_ft.json             0.1 MB
  test_probs_stageA.